In [1]:
import pandas as pd
from openai import OpenAI
from tqdm import tqdm

In [ ]:

client = OpenAI(base_url="http://localhost:1234/v1", api_key="lm-studio")
MODELO_GEN = "lmstudio-community/Qwen3.5-9B-GGUF"

def gerar_relatorios_em_lote(csv_entrada, n_amostras=100):
    df = pd.read_csv(csv_entrada, sep=',', on_bad_lines='skip')
    df_sample = df.sample(n=min(n_amostras, len(df)), random_state=42)
    
    lista_relatorios = []

    print(f"Gerando relatórios para {len(df_sample)} registros...")

    for _, row in tqdm(df_sample.iterrows(), total=len(df_sample)):
        # Preparação do Contexto Extraído do seu Dataset (uso de row[coluna], não row inteiro)
        contexto_fato = (
            f"CODIGO: {row.get('CODIGO', '')} | "
            f"NATUREZA: {row.get('NATUREZA_OCORRENCIA', '')} | "
            f"DATA: {row.get('DATA_FATO', '')} | "
            f"HORA: {row.get('HORA_FATO', '')} | "
            f"LOCAL: {row.get('MUNICIPIO', '')}/{row.get('BAIRRO', '')} | "
            f"TIPO LOCAL: {row.get('TIPO_LOCAL_FATO', '')} | "
            f"NARRATIVA: {row.get('NARRATIVA', '')}"
        )
        
        prompt_especifico = (
            f"Escreva um RELATÓRIO TÉCNICO RESUMIDO baseado nos dados abaixo:\n{contexto_fato}\n\n"
            "Siga rigorosamente este formato:\n"
            "**RELATÓRIO TÉCNICO RESUMIDO – BOLETIM DE OCORRÊNCIA**\n"
            "**DATA DOS FATOS:**...\n"
            "**HORA INICIAL DOS FATOS:**...\n"
            "**LOCAL:**...\n"
            "**NATUREZA DA OCORRÊNCIA:**...\n"
            "**RELATO SUCINTO DOS ACONTECIMENTOS**:..."
        )

        try:
            res = client.chat.completions.create(
                model=MODELO_GEN,
                messages=[{"role": "user", "content": prompt_especifico}],
                temperature=0.7
            )
            relatorio = res.choices[0].message.content
            
            lista_relatorios.append({
                "codigo_bo": row.get('CODIGO', ''),
                "narrativa_original": row.get('NARRATIVA', ''),
                "contexto_completo": contexto_fato,
                "relatorio_ia": relatorio
            })
        except Exception as e:
            print(f"Erro no BO {row.get('CODIGO', '')}: {e}")
            continue

    df_final = pd.DataFrame(lista_relatorios)
    df_final.to_csv("dataset_com_relatorios.csv", index=False, encoding='utf-8')
    print("\nEtapa de Geração Concluída.")

In [ ]:
gerar_relatorios_em_lote("Dataset/dataset_tratado.csv")

Gerando relatórios para 100 registros...


  1%|          | 1/100 [03:19<5:29:25, 199.65s/it]

Erro no BO 1997: 'list' object has no attribute 'message'


  2%|▏         | 2/100 [06:33<5:20:21, 196.14s/it]

Erro no BO 1578: 'list' object has no attribute 'message'


...

Erro no BO 7703: 'list' object has no attribute 'message'


  5%|▌         | 5/100 [16:21<5:12:43, 197.52s/it]

Erro no BO 13564: 'list' object has no attribute 'message'


  6%|▌         | 6/100 [19:49<5:14:37, 200.82s/it]

Erro no BO 2881: 'list' object has no attribute 'message'
